In [114]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

In [115]:
data1 = r"G:\My Drive\New_Working_Files\KNIME\CTU_2026\Kapture_Reports\Outputs\PL_Outputs.xlsx"
df= pd.read_excel(data1)

In [117]:
print(df.head(5))

  crm_flag       Date     Month  Week                            case_id  \
0     SRMS 2026-01-30   January     5  FLRF-CH8IKR-ZX5RGW-HP6ZDE-WRY4GPF   
1     SRMS 2026-03-13     March    11  FLRM-102NOF-TPBM1B-MED7Y4-L87WZDP   
2     SRMS 2026-02-26  February     9  FLRM-113FVD-NTLY8D-0U2EXK-PU4TK52   
3     SRMS 2026-02-26  February     9  FLRM-113FVD-NTLY8D-0U2EXK-PU4TK52   
4     SRMS 2026-02-28  February     9  FLRM-113FVD-NTLY8D-0U2EXK-PU4TK52   

                                     CallTypeDetails RegisteredBy  \
0                        To Check Status Of The Call  DRITM.19435   
1                      Delay in Service - Out of SPD  DRITM.19511   
2                      Delay in Service - Out of SPD  DRITM.19512   
3                    ST not responding/not Reachable  DRITM.19447   
4  Fake closure - Service not provided/Call close...  DRITM.19266   

        Source                                        Description  \
0  inboundCall       undefined - undefined - check status o

In [123]:
df = df[['Date', 'case_id', 'CallTypeDetails', 'Description']].copy()
df['Description'] = df['Description'].fillna('').str.replace('^undefined - undefined -', "",regex=True)

In [124]:
print(df.head(10))

        Date                            case_id  \
0 2026-01-30  FLRF-CH8IKR-ZX5RGW-HP6ZDE-WRY4GPF   
1 2026-03-13  FLRM-102NOF-TPBM1B-MED7Y4-L87WZDP   
2 2026-02-26  FLRM-113FVD-NTLY8D-0U2EXK-PU4TK52   
3 2026-02-26  FLRM-113FVD-NTLY8D-0U2EXK-PU4TK52   
4 2026-02-28  FLRM-113FVD-NTLY8D-0U2EXK-PU4TK52   
5 2026-03-04  FLRM-1FLP9T-W2DRAX-3DD7GR-X9I1POD   
6 2026-03-04  FLRM-1FLP9T-W2DRAX-3DD7GR-X9I1POD   
7 2026-02-12  FLRM-1VRY1Z-239GDK-1HF940-WDBDOXG   
8 2026-01-25  FLRM-6187PJ-HAPEG2-8PBU7B-P42SNKQ   
9 2026-01-25  FLRM-6187PJ-HAPEG2-8PBU7B-P42SNKQ   

                                     CallTypeDetails  \
0                        To Check Status Of The Call   
1                      Delay in Service - Out of SPD   
2                      Delay in Service - Out of SPD   
3                    ST not responding/not Reachable   
4  Fake closure - Service not provided/Call close...   
5                        To Check Status Of The Call   
6                      Enquiry About Service C

In [125]:
#df['Description'] = np.where(df['CallTypeDetails'].str.lower().str.strip() == 'claim registrations', 'Claim Registrations',df['Description']).copy()

In [126]:
##print(df[['CallTypeDetails', 'Description']].value_counts())

In [127]:
print(df.groupby(['CallTypeDetails','Description']).size())

CallTypeDetails                                Description                                                                                                                                                                      
Accessories Missing/Product mismatch           na - na - cx product details not matching his product is motrola but as i can see it shows realmeteclife in a replcement sheet                                                       1
Address change request denied-Non serviceable   cx s pincode is denied by st to go thr as he talked me on call and said the statement with confirning the pincode so i provided him proper tat to follow it as it was within tat    1
                                               washing machine - REALME TECHLIFE - cx says my order is today deliver  but they cannot working share tat contact with flipkart                                                       1
Address change request-Servicable               CX PROVIDE WRONG ADDRESS  AND WANT TO

In [135]:
if 'Description' in df.columns:
    df['Unified_Text'] = (df['CallTypeDetails'].fillna('').str.lower() + " " + df['Description'].fillna('').str.lower()).str.strip()
else:
    df['Unified_Text'] = df['CallTypeDetails'].fillna('').str.lower().str.strip()
    
conditions = [
    df['Unified_Text'].str.contains(r'police|court|consumer.*court|abuse|fight|die|electric.*shock|shocked|threat|station|cx.*tired|frustated|resolution|backend.*team|proper.*resolution|othervise.*complante', na=False),
    df['Unified_Text'].str.contains(r'language|tamil|telugu|telgu|kannada|malyalam|hindi|undersatnd|speaking|translate|speak|talk|comfortable|langauge|langunage|llanguage|barrier|tegelue|malyalam', na=False),
    df['Unified_Text'].str.contains(r'trimmer|grinder|mixer|cooker|iron|induction|kettle|laptop|mouse|keyboard|monitor|speaker|fan|oven|purifier|geyser|television|heater|stablizer|concentrator|thomson|sansui|marq|realme|nokia|kenstar|motorola|motolora|techlife|washing.*machine|soundbar|mixi|grander', na=False),
    df['Unified_Text'].str.contains(r'out of sp[d|p]|spd breach|sla breach|tat.*out|timeline.*breached|sdp delay|overdue|tat.*shared|pending.*month|20.*days|tat.*shared48', na=False),
    df['Unified_Text'].str.contains(r'claim.*reg|register.*claim|raised.*claim|new.*claim|process.*claim|registering.*new|complain.*confirm|agent.*register|ticket.*agent', na=False),
    df['Unified_Text'].str.contains(r'installation|cancel|request.*installation|wants.*installation|appointment.*installation|appointment.*request', na=False),
    df['Unified_Text'].str.contains(r'refund|pickup|recivd|received|return|money.*back', na=False),
    df['Unified_Text'].str.contains(r'divert|flipkart|fk team|partner.*team|jeeves|jeevs|fk side|connect.*fk|routed|center.*agent|jeevs.*mail', na=False),
    df['Unified_Text'].str.contains(r'fake.*clos|closed.*without|cancel.*without|wrongly.*clos|visit.*not.*done|not.*visited|fake.*remark|fake.*visit|fake.*promises', na=False),
    df['Unified_Text'].str.contains(r'charge|cash|bill|demanding|money|extra|cost|payment|collected|st.*solve|behaviour|rude|technician.*asking|visit.*charge', na=False),
    df['Unified_Text'].str.contains(r'check.*status|status.*call|follow up|update.*status|cx.*know|highlighted|track', na=False),
    df['Unified_Text'].str.contains(r'warranty|ew.*asked|extended warranty|amc|contract|expired|warrnty', na=False),
    df['Unified_Text'].str.contains(r'blank.*call|call.*drop|disconnect|no.*response|silent|dead.*call', na=False)
]

choices = [
    'Legal & Safety Escalation', 'Language Barrier Support', 'Product Specific / Technical Query',
    'Delay in Service - Out of SPD', 'Claim Registrations', 'Installation & Cancellation',
    'Refunds & Pickups', 'Partner Redirect (Flipkart/Jeeves)', 'Fake closure', 
    'Billing & Technician Conduct', 'To Check Status Of The Call', 'Warranty Related', 'Blank/Dropped Calls'
]

df['Category'] = np.select(conditions, choices, default = 'Uncategorized')

uncat_mask = df['Category'] == 'Uncategorized'
if uncat_mask.any():
    vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
    X = vectorizer.fit_transform(df.loc[uncat_mask, 'Unified_Text'])
    
    kmeans = KMeans(n_clusters=5, random_state=42)
    df.loc[uncat_mask, 'Cluster_label'] = kmeans.fit_predict(X).astype(str)
    
    df['Final_Type'] = np.where(df['Category'] == 'Uncategorized', 'Cluster ' + df['Cluster_label'], df['Category'])
else:
    df['Final_Type'] = df['Category']

summary =df['Final_Type'].value_counts().reset_index()
summary.columns = ['Category', 'Count']
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(2)

print("-" * 50)
print("Final Data Analysis")
print("-" * 50 )
print(summary.to_string(index=False))

def inspect_clusters(data, n_terms=8):
    print("\n" + "=" * 50)
    print("Cluster Keyword Inspection")
    print("="* 50)
    
    uncat_data = data[data['Category'] == 'Uncategorized']
    clusters = uncat_data['Cluster_label'].unique()
    
    for c in sorted(clusters):
        cluster_rows = uncat_data[uncat_data['Cluster_label']==c]
        text=cluster_rows['Unified_Text']
        cluster_size = len(cluster_rows)

        if text.str.strip().replace('', np.nan).dropna().empty:
            print(f"Cluster {c} ({cluster_size} rows) - No meaningful text found.")
            continue
        
        v= TfidfVectorizer(stop_words= 'english', ngram_range=(1,2))
        X_tfidf = v.fit_transform(text)
        
        mean_weights = np.asarray(X_tfidf.mean(axis=0).ravel())
        indices = np.argsort(mean_weights)[::-1]
        
        features = v.get_feature_names_out()

        actual_n = min(n_terms, len(features))
        top_words = [str(features[i]) for i in indices[:actual_n]]
      
        
        print(f"Cluster{c} ({cluster_size} rows) - Main Themes: {', '.join(top_words)}")

if 'uncat_mask' in locals() and uncat_mask.any():
    inspect_clusters(df)

repeat_rate = (df['case_id'].duplicated().sum()/ len(df) *100).round(2)
print(f"\nNote: Overall repeat rate is {repeat_rate}%")

output_columns = ['Date', 'case_id', 'CallTypeDetails', 'Description', 'Final_Type']
output_path = r"C:\Users\rajeshkumar.t\Desktop\Out_vas.xlsx"
df[output_columns].to_excel(output_path, index=False)

print(f"\n Success output: {output_path}")



--------------------------------------------------
Final Data Analysis
--------------------------------------------------
                          Category  Count  Percentage
               Claim Registrations  68811       34.47
Product Specific / Technical Query  54501       27.30
     Delay in Service - Out of SPD  17602        8.82
                 Refunds & Pickups  13106        6.57
       To Check Status Of The Call  11114        5.57
         Legal & Safety Escalation   8731        4.37
               Blank/Dropped Calls   6550        3.28
          Language Barrier Support   5324        2.67
                      Fake closure   5061        2.54
       Installation & Cancellation   3916        1.96
      Billing & Technician Conduct   1966        0.98
Partner Redirect (Flipkart/Jeeves)    719        0.36
                         Cluster 1    689        0.35
                         Cluster 3    493        0.25
                         Cluster 0    426        0.21
              